# Is that voice note real? Build an audio deepfake detector

**GDG On Campus · Abu Dhabi University · 11 October 2026**

In this notebook you build, test and break a voice deepfake detector, then use it on voice notes you make yourself. Everything runs on a free Colab CPU.

| Stop | What you do | What you learn |
|---|---|---|
| **Setup** | Run the first cell, read a word hidden in noise | Your runtime works; a spectrogram shows what the ear misses |
| **Stop 1 · Sound** | Listen to clips A and B, then measure them | Spectrograms, average spectra, and what file metadata can and cannot say |
| **Stop 2 · First detector** | Train a 58-feature detector, test it on eight roads | Why 100% on clean clips means nothing, and how to read an error bar |
| **Stop 3 · Detector v2** | Break the first detector with noise, then engineer a fix | 135 measurements of the voice, training on simulated roads, EER |
| **Stop 4 · Your clip** | Upload your own real and AI voice notes | Writing a report that someone else can check |

**How to work.** Run the cells in order with **Shift + Enter**. Read the text above each cell before you run it: it tells you what to predict. If something breaks, use **Runtime → Restart session and run all**.

**The data.** 60 short English clips: 30 people reading sentences (Google FLEURS) and the same 30 sentences spoken by three text-to-speech programs (Microsoft Edge TTS, Google gTTS, Piper), 10 each. Clip `real_06` and clip `ai_06_edge` say the same sentence.

**How to use this notebook tonight.** File → Save a copy in Drive, then Runtime → Run all. It takes about five minutes; leave it running while the talk goes on. You do not need to write or change any code. At each Colab stop, scroll down to that section and read the results. At Stop 4 a **Choose files** button waits for your two voice notes.

In [ ]:
#@title Setup check: run this first (about 20 seconds)
import subprocess, sys, shutil, os, time, glob, json, struct, hashlib, warnings
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "praat-parselmouth"], check=False)
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import stft, istft
from IPython.display import Audio, display
import parselmouth
warnings.filterwarnings("ignore")
assert shutil.which("ffmpeg"), "ffmpeg is missing: please use a Google Colab runtime"
encoders = subprocess.run(["ffmpeg", "-hide_banner", "-encoders"], capture_output=True, text=True).stdout
assert "libopus" in encoders and "libmp3lame" in encoders, "this ffmpeg cannot write Opus or MP3"
SR = 16000
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})

def hidden_word_hiss(word, seconds=4.0):
    """Noise whose spectrogram spells a word: louder noise where the letters are."""
    fig = plt.figure(figsize=(4, 1), dpi=100)
    fig.text(0.5, 0.5, word, ha="center", va="center", fontsize=64, weight="bold")
    fig.canvas.draw(); ink = np.asarray(fig.canvas.buffer_rgba())[:, :, 0] < 128; plt.close(fig)
    n_frames = int(seconds * SR / 128) + 1
    rows = np.linspace(0, ink.shape[0] - 1, 180).astype(int)[::-1]
    cols = np.linspace(0, ink.shape[1] - 1, n_frames).astype(int)
    mag = np.full((257, n_frames), 0.05); mag[40:220] += 0.6 * ink[np.ix_(rows, cols)]
    phase = np.exp(2j * np.pi * np.random.default_rng(7).random(mag.shape))
    _, y = istft(mag * phase, fs=SR, nperseg=512, noverlap=384)
    return 0.5 * y / np.abs(y).max()

hiss = hidden_word_hiss("GDG")
display(Audio(hiss, rate=SR))
f, t, Z = stft(hiss, fs=SR, nperseg=512, noverlap=384)
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.pcolormesh(t, f / 1000, 20 * np.log10(np.abs(Z) + 1e-9), cmap="gray_r", shading="auto")
ax.set(xlabel="time (s)", ylabel="frequency (kHz)", title="The hiss you just heard, drawn as a spectrogram")
plt.show()
print("Setup works. Type the word you can read in the Teams chat.")

## The audio bundle

Run the next cells. Their long text is the 60 clips packed as base64, so the notebook needs no download and no Google Drive.

In [ ]:
#@title Audio bundle, part 1 of 8
PART_0 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 2 of 8
PART_1 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 3 of 8
PART_2 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 4 of 8
PART_3 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 5 of 8
PART_4 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 6 of 8
PART_5 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 7 of 8
PART_6 = (
    "..."
)

In [ ]:
#@title Audio bundle, part 8 of 8
PART_7 = (
    "..."
)

In [ ]:
#@title Unpack the clips
import base64, io, zipfile
blob = base64.b64decode("".join(globals()[f"PART_{k}"] for k in range(8)))
zipfile.ZipFile(io.BytesIO(blob)).extractall(".")
shutil.copy("clips/real_06.opus", "A.ogg"); shutil.copy("clips/ai_06_edge.opus", "B.ogg")
print(len(glob.glob("clips/*.opus")), "clips unpacked, plus A.ogg and B.ogg")

In [ ]:
# A catalogue of the 60 clips. Recordings first, then AI clips, in a fixed order.
paths = sorted(glob.glob("clips/real_*.opus")) + sorted(glob.glob("clips/ai_*.opus"))
names = [os.path.basename(p) for p in paths]
label = np.array([int(n.startswith("ai_")) for n in names])                 # 0 = recorded, 1 = AI
sentence = np.array([int(n.split("_")[1].split(".")[0]) for n in names])    # which of the 30 sentences
source = np.array(["recorded" if n.startswith("real") else n.split("_")[2].split(".")[0] for n in names])
for s_, n_ in zip(*np.unique(source, return_counts=True)): print(f"{s_:9s} {n_} clips")

## Toolkit: reading audio, and the roads a clip travels

A voice note rarely reaches you as it was made. It may be played on a speaker in a room, sent over a phone call, compressed by WhatsApp or TikTok, cleaned by a noise filter, and forwarded again. Each step keeps what people hear and removes detail a detector might need.

The functions below simulate those steps with **ffmpeg**, the same encoder library many apps use:

| Function | Real-world step | What it removes or adds |
|---|---|---|
| `opus(y, kbps)` | WhatsApp voice note | everything above 8 kHz at 16 kbit/s, fine detail everywhere |
| `mp3(y, kbps)` | an MP3 re-upload | high frequencies, quiet detail under loud sounds |
| `phone(y)` | a phone call (G.711) | everything outside 300 to 3,400 Hz |
| `noise(y, snr_db)` | a café, a street | fills the silent gaps |
| `reverb(y, t60)` | a room | echo that smears pauses and breaths |
| `speaker_mic(y)` | played on a speaker, heard by a phone microphone | low bass and top treble |
| `denoise(y)` | the noise filter in phones and Teams | background noise, but also breath and room sound |

`apply_condition` chains them into the eight **test conditions** used all through this notebook. `random_channel` draws a random road for training, in Stop 3.

In [ ]:
def run_ffmpeg(args, data):
    p = subprocess.run(["ffmpeg", "-hide_banner", "-loglevel", "error", *args], input=data, capture_output=True)
    if p.returncode != 0: raise RuntimeError(p.stderr.decode()[-400:])
    return p.stdout

def decode(raw, sr=SR):
    """Any audio file (bytes) -> mono float samples at sr."""
    return np.frombuffer(run_ffmpeg(["-i", "pipe:0", "-ac", "1", "-ar", str(sr), "-f", "f32le", "pipe:1"], raw), "<f4").astype(np.float64)

def wav_bytes(y, sr=SR):
    import wave
    b = io.BytesIO(); w = wave.open(b, "wb"); w.setnchannels(1); w.setsampwidth(2); w.setframerate(sr)
    w.writeframes((np.clip(y, -1, 1) * 32767).astype("<i2").tobytes()); w.close(); return b.getvalue()

def ffmpeg_through(y, encode_args, fmt):
    """Encode y with ffmpeg, then decode it back: one trip through a codec."""
    z = decode(run_ffmpeg(["-i", "pipe:0", *encode_args, "-f", fmt, "pipe:1"], wav_bytes(y)))
    n = min(len(z), len(y)); return z[:n] if n > 0 else y

def peak_norm(y, peak=0.7): return y / (np.abs(y).max() + 1e-9) * peak
def load(path): return peak_norm(decode(open(path, "rb").read()))

from scipy.signal import butter, sosfilt, fftconvolve
def opus(y, kbps=16): return ffmpeg_through(y, ["-ar", "16000", "-c:a", "libopus", "-b:a", f"{kbps}k"], "ogg")
def mp3(y, kbps=32): return ffmpeg_through(y, ["-ar", "22050", "-c:a", "libmp3lame", "-b:a", f"{kbps}k"], "mp3")
def phone(y):
    z = sosfilt(butter(4, [300, 3400], btype="band", fs=SR, output="sos"), y)
    return ffmpeg_through(z, ["-ar", "8000", "-c:a", "pcm_mulaw"], "wav")
def noise(y, snr_db, rng, colour="pink"):
    n = rng.normal(0, 1, len(y))
    if colour == "pink":                                   # more energy at low frequencies, like traffic
        f = np.fft.rfftfreq(len(y), 1 / SR); N_ = np.fft.rfft(n); N_[1:] /= np.sqrt(f[1:]); N_[0] = 0; n = np.fft.irfft(N_, len(y))
    elif colour == "babble":                               # speech-band noise that rises and falls, like a café
        n = sosfilt(butter(2, [200, 3000], btype="band", fs=SR, output="sos"), n) * (1 + 0.8 * np.sin(2 * np.pi * 3.5 * np.arange(len(y)) / SR))
    n *= np.sqrt(np.mean(y ** 2) / 10 ** (snr_db / 10) / (np.mean(n ** 2) + 1e-12))   # SNR = 10 log10(Ps / Pn)
    return y + n
def reverb(y, t60, rng):
    t = np.arange(int(min(1.0, t60 * 1.2) * SR)) / SR
    h = rng.normal(0, 1, len(t)) * np.exp(-6.9 * t / t60); h[0] = 1.0 / (np.abs(h).sum() / 8)   # decays 60 dB in t60 seconds
    h /= np.abs(h).sum() / 8
    return fftconvolve(y, h)[:len(y)]
def speaker_mic(y, rng):
    lo, hi = rng.uniform(120, 250), rng.uniform(5500, 7600)
    return sosfilt(butter(2, [lo, hi], btype="band", fs=SR, output="sos"), y)
def denoise(y): return ffmpeg_through(y, ["-af", "afftdn=nr=30:nf=-50", "-ar", "16000", "-c:a", "pcm_s16le"], "wav")

TEST_CONDITIONS = ["clean", "opus16", "mp3_32", "phone", "noise15", "reverb", "denoise", "replay"]
NICE = {"clean": "clean", "opus16": "Opus 16 kbit/s", "mp3_32": "MP3 32 kbit/s", "phone": "phone call",
        "noise15": "café noise 15 dB", "reverb": "room echo", "denoise": "noise filter", "replay": "replay + WhatsApp"}
def apply_condition(y, cond, seed=0):
    rng = np.random.default_rng(seed)
    if cond == "clean": return y
    if cond == "opus16": return opus(y, 16)
    if cond == "mp3_32": return mp3(y, 32)
    if cond == "phone": return phone(y)
    if cond == "noise15": return noise(y, 15, rng, "babble")
    if cond == "reverb": return reverb(y, 0.5, rng)
    if cond == "denoise": return denoise(y)
    if cond == "replay":   # played on a speaker in a room, recorded by a phone, sent on WhatsApp
        z = speaker_mic(y, rng); z = reverb(z, 0.45, rng); z = noise(z, 25, rng, "pink"); return opus(peak_norm(z), 16)
    raise ValueError(cond)

def random_channel(y, rng):
    """A random road, used only for training (Stop 3). Applied to recordings and AI clips alike."""
    z = y
    if rng.random() < 0.4: z = speaker_mic(z, rng)
    if rng.random() < 0.4: z = reverb(z, rng.uniform(0.15, 0.7), rng)
    if rng.random() < 0.5: z = noise(z, rng.uniform(8, 30), rng, rng.choice(["pink", "babble", "white"]))
    if rng.random() < 0.2: z = denoise(peak_norm(z))
    r = rng.random()
    if r < 0.35: z = opus(peak_norm(z), int(rng.choice([8, 12, 16, 24])))
    elif r < 0.55: z = mp3(peak_norm(z), int(rng.choice([24, 32, 64])))
    elif r < 0.65: z = phone(peak_norm(z))
    return peak_norm(z)

def spectrogram(ax, y, title, vmax=None, top_khz=8):
    """Greyscale spectrogram in the style of Praat: 25 ms windows every 10 ms, 55 dB range."""
    y = np.append(y[0], y[1:] - 0.97 * y[:-1])                    # pre-emphasis lifts the high frequencies
    f, t, Z = stft(y, fs=SR, nperseg=400, noverlap=240, nfft=1024)
    db = 20 * np.log10(np.abs(Z) + 1e-10); vmax = db.max() if vmax is None else vmax
    ax.pcolormesh(t, f / 1000, db, vmin=vmax - 55, vmax=vmax, cmap="gray_r", shading="auto", rasterized=True)
    ax.set(ylim=(0, top_khz), ylabel="kHz", title=title); return db.max()

def _call(job): fn, j, item = job; return j, fn(item)
def run_parallel(fn, items, what):
    """Run fn on every item using all CPU cores, with a progress counter."""
    from multiprocessing import get_context
    out = [None] * len(items); t0 = time.time()
    with get_context("fork").Pool(os.cpu_count() or 2) as pool:
        for done, (j, r) in enumerate(pool.imap_unordered(_call, [(fn, j, it) for j, it in enumerate(items)]), 1):
            out[j] = r; print(f"\r{what}: {done}/{len(items)}  ({time.time() - t0:.0f} s)", end="")
    print(); return out
print("toolkit ready")

---
# Stop 1 · Listen, then measure

One of these clips is a person reading aloud; the other was generated by a text-to-speech program. **Listen with headphones, then write down A or B in the chat before you run the measurements.**

In [ ]:
A, B = load("A.ogg"), load("B.ogg")
print("Clip A"); display(Audio(A, rate=SR))
print("Clip B"); display(Audio(B, rate=SR))

### Spectrograms on one scale

A spectrogram cuts the sound into 25 ms windows, every 10 ms, and measures how much energy each frequency has in each window (a short-time Fourier transform). Time runs left to right, frequency upwards, darkness is energy in decibels. Both clips are drawn on **the same colour scale**, so the same darkness means the same level.

What to look for: voiced sounds show **harmonics**, horizontal stripes spaced by the pitch; an *s* is a dark smudge above 4 kHz; a pause is a pale column. Is the background of the pauses the same in both clips?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 5.2), sharex=True, layout="constrained")
top = max(spectrogram(axes[0], A, "Clip A"), spectrogram(axes[1], B, "Clip B"))
spectrogram(axes[0], A, "Clip A", vmax=top); spectrogram(axes[1], B, "Clip B", vmax=top)
axes[1].set_xlabel("time (s)"); plt.show()

### What the file itself says

`ffprobe` reads the container. Opus files also carry an **input-rate field**: the sample rate of the audio before it was encoded. The bitrate is $8 \times \text{bytes} / \text{seconds}$.

In [ ]:
def file_info(path):
    raw = open(path, "rb").read()
    meta = json.loads(subprocess.run(["ffprobe", "-v", "error", "-show_format", "-show_streams", "-of", "json", path],
                                     capture_output=True, text=True).stdout)
    st, fm = meta["streams"][0], meta["format"]
    info = {"bytes": len(raw), "seconds": round(float(fm.get("duration", st.get("duration", 0))), 2),
            "container": fm["format_name"], "codec": st["codec_name"], "channels": st.get("channels"),
            "decoder clock (Hz)": int(st.get("sample_rate", 0))}
    k = raw.find(b"OpusHead")                         # Opus header: input sample rate at byte 12
    if k >= 0: info["input-rate field (Hz)"] = struct.unpack("<I", raw[k + 12:k + 16])[0]
    tags = {**fm.get("tags", {}), **st.get("tags", {})}
    info["encoder"] = next((v for kk, v in tags.items() if kk.lower() == "encoder"), "not recorded")
    info["bitrate (kbit/s)"] = round(8 * len(raw) / max(info["seconds"], 1e-3) / 1000, 1)
    info["SHA-256"] = hashlib.sha256(raw).hexdigest()[:16] + "…"
    return info

for p in ["A.ogg", "B.ogg"]:
    print(p); [print(f"   {k:22s} {v}") for k, v in file_info(p).items()]

Both files went through **the same last encoder**, so the metadata names the last program that touched them, not the source of the voice. Metadata is easy to strip and easy to fake: it describes the road, not the speaker.

### The average spectrum

Decoding at 48 kHz shows how far up each file has energy. The **highest surviving frequency** tells you about the road a clip travelled: a phone call stops near 4 kHz, a WhatsApp voice note near 8 kHz.

In [ ]:
from scipy.signal import welch
def average_spectrum(path, sr=48000):
    y = decode(open(path, "rb").read(), sr); f, p = welch(y, fs=sr, nperseg=4096)
    return f, 10 * np.log10(p + 1e-20)
def highest_frequency(f, db):
    """Where the energy falls off a cliff: the largest drop over 600 Hz, above 2 kHz."""
    k = max(1, int(round(400 / (f[1] - f[0])))); sm = np.convolve(db, np.ones(k) / k, "same")
    ref = np.median(sm[(f > 300) & (f < 3000)]); step = max(1, int(round(600 / (f[1] - f[0]))))
    drop = sm[:-step] - sm[step:]
    ok = (f[:-step] > 2000) & (sm[:-step] > ref - 50) & (f[:-step] < f[-1] - 1500)
    return f[int(np.argmax(np.where(ok, drop, -1e9)))] + 300

fig, ax = plt.subplots(figsize=(8, 3))
for p, col in [("A.ogg", "k"), ("B.ogg", "0.6")]:
    f, db = average_spectrum(p); ax.plot(f / 1000, db - db.max(), color=col, lw=1, label=p)
    print(f"{p}: energy stops near {highest_frequency(f, db) / 1000:.1f} kHz")
ax.set(xlabel="frequency (kHz)", ylabel="level (dB)", xlim=(0, 24), ylim=(-110, 5)); ax.legend(); plt.show()

**Chat:** name one difference between A and B that you can *measure*, not only hear.

**Reveal:** A is a person from the FLEURS dataset; B was generated by Microsoft Edge text-to-speech. Look at the pauses: in B they are digital silence, in A they hold room noise. Keep that in mind: it is a clue about how each file was *made*, not about who is speaking. Stop 3 shows why that matters.

### Extension while you wait: make a vowel from a buzz

The Voice section explains speech as a **source** (the buzz of the vocal folds) shaped by a **filter** (the resonances of the throat and mouth). You can build both in a few lines. The source is a click repeated $F_0$ times a second; the filter is three resonators placed at the formants of a vowel.

In [ ]:
from scipy.signal import lfilter
def buzz(f0, seconds=0.8):
    x = np.zeros(int(seconds * SR)); x[::int(SR / f0)] = 1.0          # one click per vocal-fold cycle
    return lfilter([1.0], [1.0, -0.97], x)                             # soften: real pulses are not sharp clicks
def formant_filter(x, formants, bandwidths=(80, 100, 120)):
    for F, Bw in zip(formants, bandwidths):                            # one resonator per formant
        r, th = np.exp(-np.pi * Bw / SR), 2 * np.pi * F / SR
        x = lfilter([1 - r], [1, -2 * r * np.cos(th), r * r], x)
    return x / np.abs(x).max() * 0.6
VOWELS = {"aa (hod)": (730, 1090, 2440), "ee (heed)": (270, 2290, 3010), "oo (who'd)": (300, 870, 2240)}
for name, F in VOWELS.items():
    print(name, "at 120 Hz"); display(Audio(formant_filter(buzz(120), F), rate=SR))
print("aa at 220 Hz: same filter, higher source"); display(Audio(formant_filter(buzz(220), VOWELS["aa (hod)"]), rate=SR))

Same buzz, three filters: three vowels. Same filter, two buzzes: one vowel at two pitches. A speech generator has to get both parts right, and keep them consistent with one body.

---
# Stop 2 · The first detector

### 58 numbers per clip

A detector needs numbers, not sound. The first detector describes each clip with 58 numbers:

* **13 mel-frequency cepstral coefficients (MFCCs)** for every 25 ms frame. They summarise the shape of the spectrum the way the ear spaces frequencies (the mel scale). Over the clip we keep 4 statistics of each: median, standard deviation, 10th and 90th percentile, so $13 \times 4 = 52$ numbers.
* **Spectral flatness** (how noise-like each frame is), **spectral centroid** (where its energy sits) and **spectral flux** (how fast the spectrum changes): median and standard deviation of each, 6 numbers.

The MFCC of frame $t$ is a discrete cosine transform of the log energies $E_m(t)$ in 26 mel bands:
$$c_k(t) = \sum_{m=1}^{26} \log E_m(t)\,\cos\!\Big(\frac{\pi k}{26}\big(m - \tfrac12\big)\Big), \qquad k = 1, \dots, 13$$

In [ ]:
from scipy.fft import dct
def frames_db(y, win=400, hop=160):
    p = np.pad(y, (win // 2, win // 2)); fr = np.lib.stride_tricks.sliding_window_view(p, win)[::hop]
    return 20 * np.log10(np.sqrt(np.mean(fr * fr, axis=1)) + 1e-10)

def speech_span(y):
    """Cut the silence before the first and after the last loud frame."""
    y = y / (np.sqrt(np.mean(y * y)) + 1e-10) * 0.05; lev = frames_db(y); act = np.flatnonzero(lev > lev.max() - 25)
    b, e = int(act[0] * 160), int(min(len(y), (act[-1] + 1) * 160)); return y[b:e]

_mel = lambda f: 2595 * np.log10(1 + f / 700); _imel = lambda m: 700 * (10 ** (m / 2595) - 1)

def features58(y):
    sp = speech_span(y)
    _, _, z = stft(sp, fs=SR, nperseg=400, noverlap=240, boundary=None, padded=False)
    hz = np.linspace(0, SR / 2, 201)
    pts = _imel(np.linspace(_mel(80), _mel(7600), 28))                      # 26 triangular mel bands
    bank = np.array([np.maximum(0, np.minimum((hz - lo) / (mid - lo), (hi - hz) / (hi - mid))) for lo, mid, hi in zip(pts[:-2], pts[1:-1], pts[2:])])
    power = np.abs(z) ** 2 + 1e-10; logmel = np.log(bank @ power + 1e-10)
    cep = dct(logmel, axis=0, norm="ortho")[1:14]                           # 13 MFCCs per frame
    a = np.abs(z) + 1e-9
    flat = np.exp(np.mean(np.log(a), 0)) / np.mean(a, 0)                    # geometric / arithmetic mean
    cen = np.sum(np.arange(201)[:, None] * 40 * a, 0) / np.sum(a, 0)        # centre of mass in Hz
    flux = np.mean(np.abs(np.diff(logmel, axis=1)), 0)
    return np.r_[np.median(cep, 1), np.std(cep, 1), np.quantile(cep, .1, 1), np.quantile(cep, .9, 1),
                 np.median(flat), np.std(flat), np.median(cen), np.std(cen), np.median(flux), np.std(flux)]

print("clip A ->", features58(A).shape[0], "numbers; first five:", np.round(features58(A)[:5], 2))

### Eight versions of every clip

To test honestly, we need clips the detector has never seen, and roads it has never seen. The next cell sends every clip down each of the eight test roads from the toolkit: $60 \times 8 = 480$ versions. It uses both CPU cores and takes **about a minute**.

In [ ]:
def eight_versions(i):
    y = load(paths[i])
    return [apply_condition(y, c, seed=1000 + i) for c in TEST_CONDITIONS]

VERSIONS = run_parallel(eight_versions, list(range(len(paths))), "clips sent down 8 roads")
F58 = {c: np.array([features58(VERSIONS[i][k]) for i in range(len(paths))]) for k, c in enumerate(TEST_CONDITIONS)}
print("58 features for", len(paths), "clips x", len(TEST_CONDITIONS), "conditions")

### Train, test, and keep sentences apart

**Logistic regression** gives every feature a weight $w_j$, adds them up and squashes the sum into a probability:
$$p(\text{AI}) = \sigma\Big(b + \sum_j w_j x_j\Big), \qquad \sigma(z) = \frac{1}{1 + e^{-z}}$$
Features are first **standardised** (mean 0, standard deviation 1) so that no feature wins just because of its units.

**Grouped folds.** The 30 sentences are split into 5 folds of 6 (sentences 0, 5, 10, … form fold 0). We train on 4 folds and test on the fifth, five times, so every clip gets a score from a model that never heard its sentence, in any version. If a sentence appeared in both training and testing, the model could recognise the words instead of the voice.

**Predict first:** what accuracy will the first detector get on the clean clips?

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def logreg(C): return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
fold_by_sentence = sentence % 5
# unseen voice program: hold out one TTS program and 10 recorded sentences together
fold_by_program = np.array([{"edge": 0, "gtts": 1, "piper": 2}[s_] if s_ != "recorded" else min(n_ // 10, 2) for s_, n_ in zip(source, sentence)])

def first_detector_scores(folds):
    scores = {c: np.zeros(len(paths)) for c in TEST_CONDITIONS}
    for g in np.unique(folds):
        test, train = folds == g, folds != g
        model = logreg(0.3).fit(F58["clean"][train], label[train])          # trained on clean clips only
        for c in TEST_CONDITIONS: scores[c][test] = model.predict_proba(F58[c][test])[:, 1]
    return scores

S_first = first_detector_scores(fold_by_sentence)
called_ai = S_first["clean"] >= 0.5
print("Clean clips, unseen sentences:")
print(f"                 called recorded   called AI")
print(f"   recorded      {np.sum(~called_ai & (label == 0)):8d}       {np.sum(called_ai & (label == 0)):8d}")
print(f"   AI            {np.sum(~called_ai & (label == 1)):8d}       {np.sum(called_ai & (label == 1)):8d}")

A perfect score on clean clips is where the questions start. All 30 recordings come from one dataset and all 30 AI clips from three programs: the model may be recognising *the source*, not the voice. So we test it on the roads.

### Error bars: the Wilson interval

With 30 clips, one mistake moves the error rate by 3.3 points. The **95% Wilson interval** shows the range of true error rates that fit what we measured, for $k$ errors out of $n$, $\hat p = k/n$ and $z = 1.96$:
$$\frac{\hat p + \frac{z^2}{2n} \pm z\sqrt{\frac{\hat p(1-\hat p)}{n} + \frac{z^2}{4n^2}}}{1 + z^2/n}$$

In [ ]:
def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n; c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return p, max(0, c - h), min(1, c + h)

def error_table(scores, title):
    rows = []
    for c in TEST_CONDITIONS:
        p = scores[c]
        rows.append((NICE[c], int(np.sum((p < 0.5) & (label == 1))), int(np.sum((p >= 0.5) & (label == 0)))))
    fig, ax = plt.subplots(figsize=(8, 3.6), layout="constrained")
    for i, (name, miss, fa) in enumerate(rows):
        for k, n, col, dy in [(miss, 30, "tab:red", -0.15), (fa, 30, "tab:blue", 0.15)]:
            p, lo, hi = wilson(k, n); ax.plot([lo * 100, hi * 100], [i + dy] * 2, color=col, lw=2.2); ax.plot(p * 100, i + dy, "o", color=col, ms=5)
        ax.text(103, i, f"{miss}/30 missed · {fa}/30 flagged", va="center", fontsize=8.5)
    ax.set_yticks(range(len(rows)), [r[0] for r in rows]); ax.invert_yaxis(); ax.set_xlim(-2, 150); ax.set_xticks([0, 25, 50, 75, 100])
    ax.set_xlabel("error rate (%), 95% Wilson interval   ·   red: AI clips missed   ·   blue: recordings flagged as AI"); ax.set_title(title, loc="left")
    plt.show()

error_table(S_first, "First detector, unseen sentences, trained on clean clips")

In [ ]:
S_first_prog = first_detector_scores(fold_by_program)
for name, g in [("Edge", 0), ("gTTS", 1), ("Piper", 2)]:
    m = (fold_by_program == g) & (label == 1)
    print(f"Voice program never seen in training: {name:5s}  AI clips missed {np.sum(S_first_prog['clean'][m] < 0.5)}/10 (clean)")

**Chat:** which condition would worry you most for a WhatsApp voice note? Notice that a *noise filter* makes the detector flag real people, and *café noise* makes it miss AI clips. Why would noise make an AI clip look human?

---
# Stop 3 · Break it, then engineer a fix

### The shortcut

Every recording in training had a little room noise in its pauses, and every AI clip had digital silence. The first detector learned **“noise means human”**: a shortcut that is true in this dataset and false in the world. Add café noise to an AI clip and the shortcut flips the answer.

In [ ]:
B_noisy = apply_condition(B, "noise15", seed=3)
fig, axes = plt.subplots(2, 1, figsize=(10, 4.6), sharex=True, layout="constrained")
top = spectrogram(axes[0], B, "AI clip B as generated")
spectrogram(axes[1], B_noisy, "The same clip plus café noise at 15 dB SNR: the silent gaps are filled", vmax=top)
axes[1].set_xlabel("time (s)"); plt.show()
model = logreg(0.3).fit(F58["clean"][sentence != 6], label[sentence != 6])     # never saw sentence 6
for name, y in [("B as generated", B), ("B plus café noise", B_noisy)]:
    print(f"{name:20s} p(AI) = {model.predict_proba([features58(y)])[0, 1]:.2f}")
display(Audio(B_noisy, rate=SR))

### The fix has two parts

**1. Measure the voice, not only the spectrum.** Detector v2 uses 135 numbers in seven groups. Each group connects to a part of the talk:

| Group | Numbers | What it measures | Why it can separate |
|---|---|---|---|
| MFCC | 26 | spectral envelope on the mel scale (mean, sd of 13) | the vocal tract shape |
| **LFCC** | 40 | the same with **linear** frequency bands (mean, sd of 20) | keeps detail at high frequencies, where vocoders leave traces |
| LFCC change | 40 | frame-to-frame change of the LFCCs | how smoothly the spectrum moves |
| spectral shape | 15 | energy in 4 bands, flatness, centroid, flux | vocoder energy, noise-like frames |
| pauses and breaths | 5 | pauses per second, pause length, breath-like gaps per minute (a crude counter) | people breathe 8 to 14 times a minute; many generators do not |
| rhythm | 2 | syllable rate from the loudness envelope | timing of speech |
| voicing | 7 | jitter, shimmer, harmonics-to-noise ratio, pitch range and slope, voiced share | the vocal folds never repeat a cycle exactly |

**2. Train on the roads.** During training every clip is copied **six times through random roads**: codec, noise, room, phone band, noise filter. Recordings and AI clips get the same treatment, so the road stops carrying any information about the label. This is the shortcut lesson turned into engineering, and it is how published detectors are trained (RawBoost, Tak et al., ICASSP 2022).

#### Spectral features: MFCC, LFCC and spectral shape

MFCC bands are narrow at low frequencies and wide at high ones, like the ear. **LFCC** bands are equally wide from 0 to 8 kHz, so they keep the detail above 4 kHz that the ear ignores and that vocoders get wrong (Sahidullah et al., Interspeech 2015).

In [ ]:
def tri_bank(edges_hz, nfft=512):
    hz = np.linspace(0, SR / 2, nfft // 2 + 1)
    return np.array([np.maximum(0, np.minimum((hz - lo) / (mid - lo), (hi - hz) / (hi - mid))) for lo, mid, hi in zip(edges_hz[:-2], edges_hz[1:-1], edges_hz[2:])])
MEL = tri_bank(_imel(np.linspace(_mel(60), _mel(7800), 28)))      # 26 mel bands
LIN = tri_bank(np.linspace(0, 8000, 32))                            # 30 linear bands
def stats(M): return np.r_[M.mean(1), M.std(1)]                     # mean and sd of every row

def spectral_features(sp):
    _, _, z = stft(sp, fs=SR, nperseg=400, noverlap=240, nfft=512, boundary=None, padded=False)
    P = np.abs(z) ** 2 + 1e-12
    mfcc = dct(np.log(MEL @ P), axis=0, norm="ortho")[1:14]
    lfcc = dct(np.log(LIN @ P), axis=0, norm="ortho")[1:21]
    a = np.sqrt(P); f = np.linspace(0, SR / 2, P.shape[0]); tot = P.sum(0)
    bands = [P[(f >= lo) & (f < hi)].sum(0) / tot for lo, hi in [(0, 1000), (1000, 2000), (2000, 4000), (4000, 8000)]]
    flat = np.exp(np.mean(np.log(a), 0)) / np.mean(a, 0)
    cen = (f[:, None] * a).sum(0) / a.sum(0)
    flux = np.r_[0, np.mean(np.abs(np.diff(np.log(P), axis=1)), 0)]
    hb = 10 * np.log10(P[f >= 4000].sum(0))
    shape = np.r_[[b.mean() for b in bands], [b.std() for b in bands], np.median(flat), flat.std(), np.median(cen), cen.std(), np.median(flux), flux.std(), hb.std()]
    return {"MFCC": stats(mfcc), "LFCC": stats(lfcc), "LFCC change": stats(np.diff(lfcc, axis=1)), "spectral shape": shape}

#### Voicing: the vocal folds, measured with Praat

Praat finds every glottal cycle (one opening and closing of the vocal folds). **Jitter** is the average change in period from one cycle to the next, relative to the mean period; **shimmer** is the same for the peak amplitude:
$$\text{jitter} = \frac{\frac{1}{N-1}\sum_{i=1}^{N-1} |T_{i+1} - T_i|}{\frac{1}{N}\sum_{i=1}^{N} T_i}, \qquad \text{shimmer} = \frac{\frac{1}{N-1}\sum_{i=1}^{N-1} |A_{i+1} - A_i|}{\frac{1}{N}\sum_{i=1}^{N} A_i}$$
The **harmonics-to-noise ratio** (HNR) compares the periodic part of the voice with the noisy part, in dB. Pitch is measured in semitones relative to the speaker's median, so men and women are comparable.

In [ ]:
from parselmouth.praat import call
def voicing_features(sp):
    try:
        s = parselmouth.Sound(sp, SR); pp = call(s, "To PointProcess (periodic, cc)", 75, 500)
        jitter = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
        shimmer = call([s, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
        hnr = call(call(s, "To Harmonicity (cc)", 0.01, 75, 0.1, 1.0), "Get mean", 0, 0)
        f0 = s.to_pitch(time_step=0.01, pitch_floor=75, pitch_ceiling=500).selected_array["frequency"]; v = f0[f0 > 0]
        if len(v) > 10:
            st = 12 * np.log2(v / np.median(v))                                  # semitones from the median
            f0sd, f0range = st.std(), np.percentile(st, 95) - np.percentile(st, 5)
            slope = np.polyfit(np.flatnonzero(f0 > 0) * 0.01, st, 1)[0]
        else: f0sd = f0range = slope = 0.0
        out = [jitter, shimmer, hnr, f0sd, f0range, slope, len(v) / max(1, len(f0))]
        return np.nan_to_num(np.array(out, float), nan=0.0)
    except Exception:
        return np.zeros(7)

#### Timing: pauses, breaths and rhythm

A **pause** is a run of at least 150 ms that is 30 dB below the loudest frames. A **breath** is a run of 150 to 800 ms that is quieter than speech but not silent, with most of its energy below 2 kHz (Layton et al., 2024, found people breathe 8 to 14 times a minute and that breaths last at least 150 ms). **Rhythm** is the strongest repetition rate of the loudness envelope between 1 and 12 Hz, close to the syllable rate.

In [ ]:
def runs(mask):
    r, c = [], 0
    for m in mask:
        if m: c += 1
        elif c: r.append(c); c = 0
    if c: r.append(c)
    return np.array(r) * 0.01                                      # frames are 10 ms apart

def timing_features(sp):
    lev = frames_db(sp, 320, 160); top = np.percentile(lev, 95)
    _, _, z = stft(sp, fs=SR, nperseg=320, noverlap=160, boundary=None, padded=False); P = np.abs(z) ** 2 + 1e-12
    n = min(len(lev), P.shape[1]); lev, P = lev[:n], P[:, :n]
    f = np.linspace(0, SR / 2, P.shape[0]); low = P[f < 2000].sum(0) / P.sum(0)
    pauses = runs(lev < top - 30); pauses = pauses[pauses >= 0.15]
    breaths = runs((lev < top - 12) & (lev >= top - 35) & (low > 0.6)); breaths = breaths[(breaths >= 0.15) & (breaths <= 0.8)]
    dur = len(sp) / SR
    timing = np.array([len(pauses) / dur, pauses.mean() if len(pauses) else 0, pauses.sum() / dur, len(breaths) / dur * 60, breaths.mean() if len(breaths) else 0])
    e = 10 ** (lev / 20); e = e - e.mean()
    F = np.abs(np.fft.rfft(e * np.hanning(len(e)), 2048)); fr = np.fft.rfftfreq(2048, 0.01); band = (fr > 1) & (fr < 12)
    rhythm = np.array([fr[band][np.argmax(F[band])], F[(fr > 2) & (fr < 8)].sum() / (F[band].sum() + 1e-9)])
    return {"pauses and breaths": timing, "rhythm": rhythm}

GROUPS = ["MFCC", "LFCC", "LFCC change", "spectral shape", "pauses and breaths", "rhythm", "voicing"]
def features(y):
    """Detector v2: 135 numbers per clip."""
    sp = speech_span(y)
    if len(sp) < 1600: sp = np.pad(sp, (0, 1600 - len(sp)))
    g = {**spectral_features(sp), **timing_features(sp), "voicing": voicing_features(sp)}
    return np.concatenate([g[k] for k in GROUPS])

# names for every number, so we can read the model later
FEATURE_NAMES = ([f"MFCC{k} {s_}" for s_ in ("mean", "sd") for k in range(1, 14)] + [f"LFCC{k} {s_}" for s_ in ("mean", "sd") for k in range(1, 21)]
    + [f"LFCC{k} change {s_}" for s_ in ("mean", "sd") for k in range(1, 21)]
    + [f"energy {b} {s_}" for s_ in ("mean", "sd") for b in ("0-1k", "1-2k", "2-4k", "4-8k")] + ["flatness median", "flatness sd", "centroid median", "centroid sd", "flux median", "flux sd", "4-8k level sd"]
    + ["pauses per s", "pause length", "pause share", "breath-like gaps per min", "breath length", "rhythm rate", "rhythm strength"]
    + ["jitter", "shimmer", "HNR", "pitch sd", "pitch range", "pitch slope", "voiced share"])
GROUP_OF = np.repeat(GROUPS, [26, 40, 40, 15, 5, 2, 7])
assert len(FEATURE_NAMES) == len(features(A)) == 135

for name, y in [("A (recorded)", A), ("B (AI)", B)]:
    x = dict(zip(FEATURE_NAMES, features(y)))
    print(f"{name:13s} jitter {x['jitter']*100:.2f}%  shimmer {x['shimmer']*100:.1f}%  HNR {x['HNR']:.1f} dB  "
          f"pauses/s {x['pauses per s']:.2f}  breath-like gaps/min {x['breath-like gaps per min']:.1f}  pitch range {x['pitch range']:.1f} st")

### Measure every clip on every road

For each of the 60 clips: the 135 features of its 8 test versions, and of 6 new random training roads. That is $60 \times 14 = 840$ feature vectors, including 360 new trips through ffmpeg. **About two to three minutes on a Colab CPU:** a good moment to read the next cell.

In [ ]:
N_ROADS = 6
def v2_features_of_clip(i):
    test = np.array([features(v) for v in VERSIONS[i]])                    # the 8 test versions (never used for training)
    y = load(paths[i])
    roads = np.array([features(random_channel(y, np.random.default_rng(10 * i + k))) for k in range(N_ROADS)])
    return test, roads

out = run_parallel(v2_features_of_clip, list(range(len(paths))), "clips measured")
F2 = {c: np.array([o[0][k] for o in out]) for k, c in enumerate(TEST_CONDITIONS)}    # 60 x 135 per condition
ROADS = np.array([o[1] for o in out])                                                 # 60 x 6 x 135
print("test features", F2["clean"].shape, "per condition; training roads", ROADS.shape)

### Train detector v2 and compare

For each fold, the first detector trains on 48 clean clips. Detector v2 trains on the same 48 clean clips **plus their 288 road copies**, with stronger regularisation ($C = 0.05$: smaller weights, a smoother boundary). Both are tested on the 8 versions of the 12 held-out clips.

We report two numbers per condition:
* **accuracy** at the threshold 0.5: how often the verdict is right;
* **equal error rate (EER)**: the error at the threshold where false alarms equal misses. It measures how well the two groups are *separated*, whatever threshold you choose.

In [ ]:
def eer(scores, labels):
    best = (1.0, 0.5)
    for t in np.r_[np.unique(scores), scores.max() + 1e-9]:
        far = np.mean(scores[labels == 0] >= t); frr = np.mean(scores[labels == 1] < t)
        if abs(far - frr) < best[0]: best = (abs(far - frr), (far + frr) / 2)
    return best[1]

def v2_scores(folds, C=0.05):
    scores = {c: np.zeros(len(paths)) for c in TEST_CONDITIONS}
    for g in np.unique(folds):
        test, train = folds == g, folds != g
        X = np.vstack([F2["clean"][train], ROADS[train].reshape(-1, ROADS.shape[2])])
        y = np.r_[label[train], np.repeat(label[train], N_ROADS)]
        model = logreg(C).fit(X, y)
        for c in TEST_CONDITIONS: scores[c][test] = model.predict_proba(F2[c][test])[:, 1]
    return scores

S_v2 = v2_scores(fold_by_sentence); S_v2_prog = v2_scores(fold_by_program)
RESULTS = {}
for proto, s1, s2 in [("unseen sentences", S_first, S_v2), ("unseen voice program", S_first_prog, S_v2_prog)]:
    print(f"\n{proto}:  accuracy (EER)")
    print(f"   {'condition':22s} {'first detector':>16s} {'detector v2':>14s}")
    for c in TEST_CONDITIONS:
        a1, a2 = np.mean((s1[c] >= 0.5) == label), np.mean((s2[c] >= 0.5) == label)
        e1, e2 = eer(s1[c], label), eer(s2[c], label); RESULTS[(proto, c)] = (a1, a2, e1, e2)
        print(f"   {NICE[c]:22s} {a1*100:6.0f}% ({e1*100:4.1f}%) {a2*100:7.0f}% ({e2*100:4.1f}%)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), sharey=True, layout="constrained")
x = np.arange(len(TEST_CONDITIONS))
for ax, proto in zip(axes, ["unseen sentences", "unseen voice program"]):
    a1 = [RESULTS[(proto, c)][0] * 100 for c in TEST_CONDITIONS]; a2 = [RESULTS[(proto, c)][1] * 100 for c in TEST_CONDITIONS]
    ax.bar(x - 0.2, a1, 0.38, color="0.75", label="first detector"); ax.bar(x + 0.2, a2, 0.38, color="tab:blue", label="detector v2")
    ax.axhline(50, color="k", lw=0.6, ls=":"); ax.set_xticks(x, [NICE[c] for c in TEST_CONDITIONS], rotation=35, ha="right"); ax.set_title(proto, loc="left")
axes[0].set_ylabel("accuracy (%)"); axes[0].set_ylim(0, 105); axes[0].legend(loc="lower left"); plt.show()

### Scores, thresholds and the three zones

A detector outputs a score; the threshold turns it into a verdict. Move the threshold left and more recordings are flagged (the **false-alarm rate** rises); move it right and more AI clips get through (the **miss rate** rises). The **equal error rate (EER)** is the rate at the threshold where the two are equal:
$$\text{false-alarm rate}(\tau) = P(s \ge \tau \mid \text{recorded}), \qquad \text{miss rate}(\tau) = P(s < \tau \mid \text{AI})$$
Instead of one threshold we report **three zones**, as forensic examiners do: below 0.3 *consistent with a recording*, 0.3 to 0.7 *inconclusive*, above 0.7 *consistent with AI*.

In [ ]:
COND = "replay"  #@param ["clean", "opus16", "mp3_32", "phone", "noise15", "reverb", "denoise", "replay"]
s = S_v2[COND]; th = np.linspace(0, 1, 201)
far = [np.mean(s[label == 0] >= t) * 100 for t in th]; frr = [np.mean(s[label == 1] < t) * 100 for t in th]
fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), layout="constrained")
axes[0].hist([s[label == 0], s[label == 1]], bins=np.linspace(0, 1, 21), color=["tab:blue", "tab:red"], label=["recorded", "AI"])
for t in (0.3, 0.7): axes[0].axvline(t, color="k", lw=0.8, ls="--")
axes[0].set(xlabel="detector v2 score p(AI)", ylabel="clips", title=f"Scores after: {NICE[COND]}"); axes[0].legend()
axes[1].plot(th, far, color="tab:blue", label="false alarms: recordings flagged"); axes[1].plot(th, frr, color="tab:red", label="misses: AI clips let through")
axes[1].set(xlabel="threshold", ylabel="error (%)", title=f"EER = {eer(s, label)*100:.1f}%"); axes[1].legend(); plt.show()
zones = np.digitize(s, [0.3, 0.7])
for z_, name in enumerate(["consistent with a recording", "inconclusive", "consistent with AI"]):
    print(f"{name:28s} recordings {np.sum((zones == z_) & (label == 0)):2d}   AI clips {np.sum((zones == z_) & (label == 1)):2d}")

### What did the detector learn?

Logistic regression is not a black box: after standardising, the size of each weight says how much that number moves the score. Summing the absolute weights per group shows which part of the voice the detector relies on.

In [ ]:
final = logreg(0.05).fit(np.vstack([F2["clean"], ROADS.reshape(-1, 135)]), np.r_[label, np.repeat(label, N_ROADS)])
w = final[-1].coef_[0]
share = {g: np.abs(w[GROUP_OF == g]).sum() for g in GROUPS}; tot = sum(share.values())
fig, ax = plt.subplots(figsize=(7, 2.8), layout="constrained")
ax.barh(GROUPS, [share[g] / tot * 100 for g in GROUPS], color="0.4"); ax.invert_yaxis(); ax.set_xlabel("share of total |weight| (%)"); plt.show()
print("Ten strongest single numbers (+ pushes towards AI, - towards recorded):")
for j in np.argsort(-np.abs(w))[:10]: print(f"   {w[j]:+.2f}  {FEATURE_NAMES[j]}")

### Base rates: what a flag means when fakes are rare

A detector's error rates are measured on a test set where half the clips are fake. In a family WhatsApp group almost none are. Bayes' theorem gives the share of flags that are really fake, for a share $\pi$ of fakes:
$$P(\text{fake} \mid \text{flag}) = \frac{\text{TPR}\,\pi}{\text{TPR}\,\pi + \text{FPR}\,(1-\pi)}$$
TPR is the share of fakes caught; FPR the share of real notes flagged. With only 30 recordings we cannot measure a small FPR precisely, so we use the top of its 95% Wilson interval: the worst rate our data cannot rule out.

In [ ]:
s = S_v2["replay"]; tpr = np.mean(s[label == 1] >= 0.5); k_fa = int(np.sum(s[label == 0] >= 0.5))
fpr_high = wilson(k_fa, 30)[2]
print(f"After replay, detector v2 catches {tpr:.0%} of fakes and flagged {k_fa} of 30 recordings;")
print(f"with 30 recordings, its true false-alarm rate could still be as high as {fpr_high:.0%}.\n")
for share in [0.5, 0.1, 0.01, 0.001]:
    ppv = tpr * share / (tpr * share + fpr_high * (1 - share))
    print(f"If {share:6.1%} of voice notes are fake: about {ppv:4.0%} of its flags are really fake")

**What we can claim.** On 60 clips, three generators and simulated roads, detector v2 holds up far better than the first detector when the road changes, and less well when the voice program is new. That is a teaching result, not a product: published detectors with about 1% error on their benchmark rose to 34 to 66% on real public-figure deepfakes (Müller et al., 2022). The professional answer to “how accurate is it?” is always *on what data, under what conditions, at what threshold?*

**Chat:** which road still hurts detector v2 the most, and why do you think so?

---
# Stop 4 · Your clip

Make **two voice notes that travel the same road**, then run both through detector v2.

1. **Real:** record yourself on WhatsApp saying: *“Hi, it’s me. I’m on a friend’s phone. Can you send me the code you just got?”*
2. **AI:** in Google AI Studio, generate the same sentence with a **stock voice** (Generate speech). Play it on your laptop speaker and record it as a WhatsApp voice note on your phone, so it takes the same road as your own voice.
3. Send both notes to yourself (WhatsApp Web, or email), download them, and upload them below.

**Rules.** Use only your own voice or AI Studio’s stock voices. Never clone another person’s voice. Colab’s terms do not allow creating deepfakes inside Colab, so generation happens in AI Studio and Colab only analyses.

First, train the final detector on all 60 clips and their road copies (a few seconds).

In [ ]:
FINAL = logreg(0.05).fit(np.vstack([F2["clean"], ROADS.reshape(-1, 135)]), np.r_[label, np.repeat(label, N_ROADS)])
REF = {k: F2["clean"][label == k] for k in (0, 1)}          # reference values from the 60 clean clips
print("final detector v2 trained on", 60 * (1 + N_ROADS), "examples")

Run the next cell and choose your files (WhatsApp `.opus` or `.ogg`, iPhone `.m4a`, `.wav`, `.mp3` all work). If the upload button does nothing, drag the files into the **Files** panel on the left and type their names in `MY_FILES` instead.

In [ ]:
MY_FILES = []   # for example ["my_voice.opus", "ai_studio.opus"]; leave empty to use the upload button
if not MY_FILES:
    try:
        from google.colab import files
        MY_FILES = list(files.upload().keys())
    except ImportError:
        MY_FILES = ["A.ogg", "B.ogg"]          # outside Colab: analyse the two workshop clips
print("files to analyse:", MY_FILES)

In [ ]:
ZONE = ["consistent with an original recording", "inconclusive", "consistent with AI generation"]
def segment_scores(y, win=3.0, hop=0.5):
    n, h = int(win * SR), int(hop * SR)
    if len(y) <= n: return np.array([len(y) / SR / 2]), FINAL.predict_proba([features(y)])[:, 1]
    starts = range(0, len(y) - n + 1, h)
    return np.array([(b + n / 2) / SR for b in starts]), FINAL.predict_proba([features(y[b:b + n]) for b in starts])[:, 1]

def report(path):
    info = file_info(path); y = load(path); x = features(y); p = FINAL.predict_proba([x])[0, 1]; zone = int(np.digitize(p, [0.3, 0.7]))
    f, db = average_spectrum(path); top = highest_frequency(f, db) / 1000
    road = ("phone call band (below 4 kHz)" if top < 4.5 else "WhatsApp or wideband call (up to 8 kHz)" if top < 8.8
            else "compressed, up to about 12 kHz" if top < 13 else "close to full band")
    t, seg = segment_scores(y); X = dict(zip(FEATURE_NAMES, x))
    def where(name):
        j = FEATURE_NAMES.index(name); r, a = REF[0][:, j], REF[1][:, j]
        return f"{X[name]:.3g} (recordings {np.median(r):.3g}, AI clips {np.median(a):.3g})"
    print("=" * 78); print(f"REPORT  {path}")
    rows = [("Material", f"{info['codec']} in {info['container']}, {info['seconds']} s, {info['bitrate (kbit/s)']} kbit/s, SHA-256 {info['SHA-256']}"),
            ("Road", f"energy up to {top:.1f} kHz: {road}; encoder tag: {info['encoder']}"),
            ("Checks", "listening, detector v2 (whole clip and 3 s windows every 0.5 s), voicing, breathing"),
            ("Score", f"p(AI) = {p:.2f}  ->  {ZONE[zone]}"),
            ("Windows", f"{len(seg)} windows: {int(np.sum(seg < 0.3))} below 0.3, {int(np.sum((seg >= 0.3) & (seg <= 0.7)))} between, {int(np.sum(seg > 0.7))} above 0.7: "
                        + ("mixed, the windows disagree" if (seg.min() < 0.3 and seg.max() > 0.7) else "the windows agree")),
            ("Voicing", f"jitter {where('jitter')}; shimmer {where('shimmer')}; HNR {where('HNR')}"),
            ("Timing", f"breath-like gaps/min {where('breath-like gaps per min')}; pauses/s {where('pauses per s')}"),
            ("Limits", "trained on 60 English clips and 3 generators; short clips are unreliable; a replayed clip carries the room's noise"),
            ("Next step", "verify through another channel: call the person back on a number you already have")]
    for k, v in rows: print(f"{k:10s} {v}")
    fig, axes = plt.subplots(2, 1, figsize=(10, 3.8), sharex=True, layout="constrained", gridspec_kw={"height_ratios": [2, 1]})
    spectrogram(axes[0], y, path); axes[1].plot(t, seg, "o-", color="k", ms=4)
    axes[1].axhspan(0, 0.3, color="tab:blue", alpha=0.12); axes[1].axhspan(0.7, 1, color="tab:red", alpha=0.12)
    axes[1].set(ylim=(0, 1), ylabel="p(AI)", xlabel="time (s)"); plt.show()
    return p

MY_SCORES = {p_: report(p_) for p_ in MY_FILES}

**Post in the chat** for each file: its zone, and whether its windows agree or are mixed (some below 0.3 and some above 0.7). The detector was trained on English and on three generators: **a wrong answer is a result, not a mistake.** Think about which part of the road could explain it.

**Compare** with a research detector: upload the same files to *Deepfake Total* (Fraunhofer AISEC, up to 90 s) and set its answer beside ours.

**Write the report** for the opening voice note: material, road, checks, results, limits, conclusion, next steps. The strongest line is usually the call back, not the score.

---
## Going further (optional, after the workshop): a self-supervised front end

The strongest lab detectors do not use hand-made features. They feed raw audio into a model pre-trained on hundreds of thousands of hours of speech (wav2vec 2.0 XLS-R: 436,000 hours, 128 languages) and train a small classifier on its internal representations (Tak et al., Odyssey 2022: 0.82% EER on ASVspoof 2021 LA).

The cell below does the same at small scale with **WavLM Base+** (Microsoft, 95 M parameters): it averages the model's layer outputs over time, then trains the same logistic regression with the same folds and roads. It downloads about 380 MB and takes **10 to 15 minutes on a CPU**, so it is off by default. With only 60 clips, expect wide error bars.

In [ ]:
#@title Self-supervised embeddings + logistic regression
RUN_SSL = False  #@param {type:"boolean"}
SSL_MODEL = "microsoft/wavlm-base-plus"  #@param ["microsoft/wavlm-base-plus", "facebook/wav2vec2-base"]
if RUN_SSL:
    import torch
    from transformers import AutoFeatureExtractor, AutoModel
    torch.set_num_threads(os.cpu_count() or 2)
    extractor = AutoFeatureExtractor.from_pretrained(SSL_MODEL); ssl = AutoModel.from_pretrained(SSL_MODEL).eval()
    def embed(y):
        with torch.no_grad():
            inp = extractor(speech_span(y).astype(np.float32), sampling_rate=SR, return_tensors="pt")
            hidden = ssl(**inp, output_hidden_states=True).hidden_states          # one tensor per layer
        return torch.stack(hidden[1:7])[:, 0].mean(dim=1).mean(dim=0).numpy()     # layers 1-6, mean over time
    t0 = time.time(); E = {}
    for k, c in enumerate(TEST_CONDITIONS):
        E[c] = np.array([embed(VERSIONS[i][k]) for i in range(len(paths))]); print(f"\rembedded {c} ({time.time() - t0:.0f} s)", end="")
    E_roads = np.array([[embed(random_channel(load(paths[i]), np.random.default_rng(10 * i + k))) for k in range(2)] for i in range(len(paths))])
    print(f"\rembeddings done in {time.time() - t0:.0f} s")
    for proto, folds in [("unseen sentences", fold_by_sentence), ("unseen voice program", fold_by_program)]:
        sc = {c: np.zeros(len(paths)) for c in TEST_CONDITIONS}
        for g in np.unique(folds):
            te, tr = folds == g, folds != g
            m = logreg(0.05).fit(np.vstack([E["clean"][tr], E_roads[tr].reshape(-1, E_roads.shape[2])]), np.r_[label[tr], np.repeat(label[tr], 2)])
            for c in TEST_CONDITIONS: sc[c][te] = m.predict_proba(E[c][te])[:, 1]
        print(f"\n{proto}: accuracy (EER), {SSL_MODEL} vs detector v2")
        for c in TEST_CONDITIONS:
            print(f"   {NICE[c]:22s} {np.mean((sc[c] >= 0.5) == label)*100:5.0f}% ({eer(sc[c], label)*100:4.1f}%)   v2 {RESULTS[(proto, c)][1]*100:4.0f}% ({RESULTS[(proto, c)][3]*100:4.1f}%)")
else:
    print("Skipped. Tick RUN_SSL and run this cell to try it.")

---
## Sources

* Data: Google FLEURS (Conneau et al., 2022); Microsoft Edge TTS, Google gTTS, Piper TTS.
* Praat (Boersma and Weenink), used through Parselmouth (Jadoul et al., 2018).
* LFCC: Sahidullah, Kinnunen and Hanilçi, *A comparison of features for synthetic speech detection*, Interspeech 2015.
* Augmentation of both classes: Tak et al., *RawBoost*, ICASSP 2022.
* Breathing: Layton et al., *Every breath you don't take: deepfake speech detection using breath*, 2024.
* Lab versus wild: Müller et al., *Does audio deepfake detection generalize?*, Interspeech 2022.
* Self-supervised front end: Tak et al., *Automatic speaker verification spoofing and deepfake detection using wav2vec 2.0 and data augmentation*, Odyssey 2022.
* Wilson interval: Wilson, *Probable inference*, JASA 1927.